# AQUASPECT — Water Quality & Inland/Coastal Water Intelligence
**Arab Youth Space Hackathon: 813 Challenge — Theme 6 PoC**

This notebook executes the end-to-end AQUASPECT analytical pipeline.
Based on data availability in the open STAC archives, this project employs a **Dual-AOI Strategy**:
1. **El Gouna (Red Sea)**: Demonstrating Planet Tanager hyperspectral capabilities (the only confirmed Egypt scene).
2. **Lake Manzala (Nile Delta)**: Demonstrating temporal anomaly detection and validation using Sentinel-2 against published in-situ records.

*Note: The code below relies on the `aquaspect` local python package located in `src/`.*

In [ ]:
import os
import sys
from pathlib import Path

# Ensure src/ is on the python path
PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Load credentials from .env (NEVER COMMIT YOUR .ENV FILE)
load_dotenv(PROJECT_ROOT / ".env")

# AQUASPECT modules
from aquaspect import config, preprocessing, indices, detection, visualization, validation
from aquaspect.data import search_sentinel2, download_s2_band, load_raster_clip

import h5py

print("AQUASPECT environment initialized successfully.")

## Phase 1: Hyperspectral Analysis (Planet Tanager)
**AOI**: El Gouna, Red Sea (Scene: `20250926_092059_95_4001`)

In [ ]:
TANAGER_FILE = PROJECT_ROOT / "data" / "sample_input" / "20250926_092059_95_4001_ortho_sr_hdf5.h5"

if not TANAGER_FILE.exists():
    print("Tanager HDF5 file not found. Please run `python run_tanager.py` to download it.")
else:
    # 1. Quality mask
    valid_mask, qa_stats = preprocessing.build_qa_mask(str(TANAGER_FILE))
    print(f"QA Stats: {qa_stats}")

    # 2. Load Required Bands
    with h5py.File(TANAGER_FILE, "r") as f:
        wavelengths = f["HDFEOS/GRIDS/HYP/Data Fields/wavelength"][:]
        sr = f["HDFEOS/GRIDS/HYP/Data Fields/surface_reflectance"]
        
        def load_b(target_nm):
            idx = np.argmin(np.abs(wavelengths - target_nm))
            arr = sr[idx, :, :].astype(np.float32) * 1e-4
            arr[~valid_mask] = np.nan
            arr[arr < 0] = np.nan
            arr[arr > 1] = np.nan
            return arr

        b_green = load_b(560)
        b_red = load_b(665)
        b_rededge = load_b(708)
        b_nir2 = load_b(860)

    # 3. Water Mask
    ndwi_arr = indices.ndwi(b_green, b_nir2)
    water_mask_raw = indices.water_mask(ndwi_arr, threshold=0.0)
    water_mask, water_stats = detection.apply_water_mask(water_mask_raw & valid_mask, min_pixels=100, pixel_area_m2=900)
    print(f"Water Stats: {water_stats}")
    
    fig1 = visualization.plot_index_map(ndwi_arr, mask=valid_mask, title="El Gouna NDWI")
    plt.show()

In [ ]:
if TANAGER_FILE.exists():
    # 4. Chlorophyll Screening
    ndci_arr = indices.ndci(b_rededge, b_red)
    chlorophyll_classes = indices.chlorophyll_screen(ndci_arr, water_mask, threshold_elevated=0.05, threshold_high=0.20)
    
    class_def = {
        0: ("Normal/Low", "#d0d0d0"),
        1: ("Elevated", "#f4a582"),
        2: ("High (Anomaly Candidate)", "#ca0020")
    }
    fig2 = visualization.plot_classified_map(chlorophyll_classes, class_def, title="Potential Chlorophyll Anomaly Screening")
    plt.show()

    # 5. Turbidity Proxy
    turb_arr = indices.turbidity_proxy(b_red, b_green)
    fig3 = visualization.plot_index_map(turb_arr, mask=water_mask, title="Turbidity Proxy (Red/Green)", cmap="YlOrBr")
    plt.show()

## Phase 2: Temporal Anomaly Detection & Validation (Sentinel-2)
**AOI**: Lake Manzala

In [ ]:
print("--- AQUASPECT Validation Report ---\n")

val_report = validation.validation_not_possible(
    reason="Lack of concurrent in-situ physical measurements for precise NTU/chlorophyll calibration.",
    attempted=["Cross-reference with CGLS LWQ 100m", "Comparison against published NIOF WQI gradients"]
)

print(json.dumps(val_report, indent=2))